# 01 — Dataset Curation Pipeline

**Project:** Diffusion LoRA Fine-Tuning on Product Images  
**Author:** Adebanji Oluwatimileyin Adelowo

This notebook walks through the complete dataset curation pipeline:
1. Image quality filtering (resolution, aspect ratio, blur)
2. Aspect-ratio normalization and center-crop
3. Near-duplicate removal via perceptual hashing
4. Background consistency check
5. BLIP-2 auto-captioning with trigger word injection
6. Train/validation split
7. Metadata export

**Output:** `data/processed/` (clean images) + `data/metadata.csv`

## Folder Structure

```
data/
├── raw/                    ← drop your source images here
├── processed/              ← pipeline output (clean, resized)
├── metadata.csv            ← captions + split assignments
└── metadata_schema.json    ← schema reference
```

### Common Mistakes to Avoid
- **Don't mix aspect ratios** without cropping — SDXL expects square or near-square at 1024px
- **Don't skip deduplication** — duplicates cause the model to overfit specific images
- **Don't use generic captions** — "a pair of glasses" teaches nothing; use structured captions
- **Don't include lifestyle images** unless you want the model to generate people too
- **Don't forget licenses** — always record image source and rights before training

In [ ]:
# Install dependencies
# !pip install Pillow imagehash transformers accelerate torch torchvision

import os
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter

try:
    import imagehash
    print("imagehash available — duplicate detection enabled")
except ImportError:
    imagehash = None
    print("WARNING: pip install imagehash for duplicate detection")

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RESOLUTION = 1024       # SDXL native; use 512 for SD 1.5
MIN_RESOLUTION = 512
MAX_ASPECT_RATIO = 1.5
BLUR_THRESHOLD = 100    # Laplacian variance — below this = blurry
HASH_THRESHOLD = 8      # pHash distance for near-duplicate detection

In [ ]:
## Stage 1: Load raw images and apply quality filters

SUPPORTED = {".jpg", ".jpeg", ".png", ".webp"}

raw_paths = [p for p in RAW_DIR.rglob("*") if p.suffix.lower() in SUPPORTED]
print(f"Found {len(raw_paths)} raw images")

def compute_blur_score(img: Image.Image) -> float:
    """Laplacian variance — higher = sharper."""
    gray = img.convert("L")
    arr = np.array(gray, dtype=np.float32)
    laplacian = np.array(Image.fromarray(arr).filter(ImageFilter.FIND_EDGES))
    return float(laplacian.var())

filtered, reasons = [], []
for path in raw_paths:
    try:
        img = Image.open(path).convert("RGB")
    except Exception as e:
        reasons.append((path.name, f"open_error: {e}"))
        continue

    if min(img.width, img.height) < MIN_RESOLUTION:
        reasons.append((path.name, f"low_res: {img.width}x{img.height}"))
        continue

    ratio = max(img.width, img.height) / min(img.width, img.height)
    if ratio > MAX_ASPECT_RATIO:
        reasons.append((path.name, f"aspect_ratio: {ratio:.2f}"))
        continue

    blur = compute_blur_score(img)
    if blur < BLUR_THRESHOLD:
        reasons.append((path.name, f"blurry: score={blur:.1f}"))
        continue

    filtered.append(path)

print(f"After quality filter: {len(filtered)} images kept, {len(reasons)} rejected")
for name, reason in reasons[:10]:
    print(f"  REJECT {name}: {reason}")

In [ ]:
## Stage 2: Near-duplicate removal

def remove_duplicates(paths: list, threshold: int) -> list:
    if imagehash is None:
        print("Skipping dedup — imagehash not available")
        return paths

    seen_hashes = {}
    unique = []
    n_dup = 0
    for path in paths:
        img = Image.open(path)
        h = imagehash.phash(img)
        if any(abs(h - sh) <= threshold for sh in seen_hashes):
            n_dup += 1
            print(f"  DUP: {path.name}")
            continue
        seen_hashes[h] = path.name
        unique.append(path)
    print(f"Duplicates removed: {n_dup}. Unique images: {len(unique)}")
    return unique

unique_paths = remove_duplicates(filtered, HASH_THRESHOLD)

In [ ]:
## Stage 3: Center-crop and resize

def center_crop_resize(img: Image.Image, size: int) -> Image.Image:
    w, h = img.width, img.height
    s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    return img.resize((size, size), Image.LANCZOS)

processed_paths = []
for path in unique_paths:
    img = Image.open(path).convert("RGB")
    processed = center_crop_resize(img, RESOLUTION)
    out = PROCESSED_DIR / path.name
    processed.save(out, quality=95)
    processed_paths.append(out)

print(f"Processed {len(processed_paths)} images → {PROCESSED_DIR}")

In [ ]:
## Stage 4: BLIP-2 auto-captioning
# Requires GPU. Skip and write captions manually if needed.

import torch
from transformers import AutoProcessor, Blip2ForConditionalGeneration

TRIGGER_WORD = "sks eyewear"
BLIP2_MODEL = "Salesforce/blip2-opt-2.7b"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained(BLIP2_MODEL)
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
blip_model.eval()

def caption_image(img: Image.Image) -> str:
    inputs = processor(images=img, return_tensors="pt").to(device, torch.float16)
    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_new_tokens=50)
    return processor.decode(ids[0], skip_special_tokens=True).strip()

captions = {}
for path in processed_paths:
    img = Image.open(path).convert("RGB")
    auto_cap = caption_image(img)
    full_cap = f"a photo of {TRIGGER_WORD} {auto_cap}"
    captions[path] = {"auto": auto_cap, "full": full_cap}
    print(f"  {path.name}: {full_cap[:80]}...")

print(f"\nCaptioned {len(captions)} images")

In [ ]:
## Stage 5: Train/val split and metadata export

VAL_FRACTION = 0.1
METADATA_PATH = Path("../data/metadata.csv")

random.seed(42)
n_val = max(1, int(len(processed_paths) * VAL_FRACTION))
val_set = set(random.sample([str(p) for p in processed_paths], n_val))

rows = []
for path in processed_paths:
    split = "val" if str(path) in val_set else "train"
    cap = captions.get(path, {"auto": "", "full": ""})
    rows.append({
        "file_path": str(path),
        "caption": cap["full"],
        "trigger_word": TRIGGER_WORD,
        "auto_caption": cap["auto"],
        "split": split,
    })

with open(METADATA_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

n_train = sum(1 for r in rows if r["split"] == "train")
n_val_actual = sum(1 for r in rows if r["split"] == "val")
print(f"Metadata written to {METADATA_PATH}")
print(f"  Train: {n_train}  |  Val: {n_val_actual}")
print("\nNEXT: Run 02_training_workflow.ipynb")